# Read the module, check the hand-note, give feedback

Five steps. AI is used in two of them and both are marked.

1. Setup
2. The module
3. Read the hand-note **(AI - needs the API key)**
4. Check it against the module *(no AI)*
5. Write the feedback **(AI - needs the API key)**

## 1. Setup

Put your key in a file called `.env` next to this notebook:

```
OPENAI_API_KEY=sk-...
```

In [1]:
#pip install openai-agents
#pip install python-dotenv

In [2]:
from agents import Agent, Runner
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
import base64, json, os, re

print("API key loaded:", bool(os.environ.get("OPENAI_API_KEY")))

API key loaded: True


## 2. The module

Copied from the task page on WesFellow Hub.

The page lists 20 tough words. The screenshot you sent cuts off after the 11th,
so we have 11. Step 4 will say so rather than pretending otherwise.

In [4]:
MODULE_TITLE   = "DAY 12 - TASK 2 - TEACHER ENGLISH SPEAKING TRAINING"
MODULE_ARTICLE = "Helping Students Overcome Fear of Speaking English"

MODULE_WORDS = {
    "fear":          "\u0921\u0930",
    "confidence":    "\u0906\u0924\u094d\u092e\u0935\u093f\u0936\u094d\u0935\u093e\u0938",
    "pronunciation": "\u0909\u091a\u094d\u091a\u093e\u0930\u0923",
    "judgment":      "\u0928\u093f\u0930\u094d\u0923\u092f",
    "environment":   "\u0935\u093e\u0924\u093e\u0935\u0930\u0923",
    "manageable":    "\u092a\u094d\u0930\u092c\u0902\u0927\u0928\u0940\u092f",
    "gradually":     "\u0927\u0940\u0930\u0947-\u0927\u0940\u0930\u0947",
    "pressure":      "\u0926\u092c\u093e\u0935",
    "encourage":     "\u092a\u094d\u0930\u094b\u0924\u094d\u0938\u093e\u0939\u093f\u0924 \u0915\u0930\u0928\u093e",
    "bravery":       "\u0938\u093e\u0939\u0938",
    "criticism":     "\u0906\u0932\u094b\u091a\u0928\u093e",
}

MODULE_WORDS_TOTAL = 20          # what the page says
MODULE_WORDS_KNOWN = len(MODULE_WORDS)

print(MODULE_TITLE)
print("Article:", MODULE_ARTICLE)
print(f"Answer key: {MODULE_WORDS_KNOWN} of {MODULE_WORDS_TOTAL} words")

DAY 12 - TASK 2 - TEACHER ENGLISH SPEAKING TRAINING
Article: Helping Students Overcome Fear of Speaking English
Answer key: 11 of 20 words


## 3. Read the hand-note  ·  **AI, needs the key**

This is the only thing here that genuinely needs a model. There is no way to read
Devanagari handwriting from a photo with plain code.

The prompt asks it to **transcribe only**. It is not asked to mark anything or
form any opinion - that happens in step 4, where you can check it.

Point `HANDNOTE` at your image. The two pages you sent are in `samples/`.

In [5]:
HANDNOTE = "samples/handnote_2.png"       # <-- change this to try another page

READ_PROMPT = """
Transcribe this handwritten homework page. Do not correct or mark anything.

Return JSON:
  "header"  - the title or module code written at the top, exactly as written
  "words"   - [{"english": "...", "hindi": "...", "clear": true/false}]
  "points"  - the numbered sentences, as written

Set "clear" to false where the writing is smudged or corrected. Say you are
unsure rather than guessing.
"""


def read_page(path):
    from openai import OpenAI
    image = base64.b64encode(open(path, "rb").read()).decode()
    reply = OpenAI().chat.completions.create(
        model="gpt-4o",
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": [
            {"type": "text", "text": READ_PROMPT},
            {"type": "image_url",
             "image_url": {"url": f"data:image/png;base64,{image}"}},
        ]}],
    )
    return json.loads(reply.choices[0].message.content)


page = read_page(HANDNOTE)

print("Header:", page["header"])
print("Words :", len(page["words"]))
print("Points:", len(page["points"]))

RateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}

## 4. Check it against the module  ·  *no AI*

Plain comparison. Three possible answers, and the first one has to be asked first:
if the page is a different module's homework, marking its words against this
module's key would give a near-zero for work the student did correctly.

In [ ]:
def tidy(s):
    """Lower-case, and drop marks that are not errors (nukta, zero-width)."""
    s = (s or "").strip().lower()
    return re.sub(r"[\u093c\u200c\u200d]", "", s)


def check(page):
    header = tidy(page["header"])

    # Does the header name a DIFFERENT module, like "71-TS"?
    other = re.search(r"\b\d{1,3}\s*-\s*[a-z]{1,3}\b", header)
    ours = [w for w in tidy(MODULE_ARTICLE).split() if len(w) > 4]
    names_ours = any(w in header for w in ours)

    if other and not names_ours:
        return {"case": "wrong module",
                "facts": [f"The page is headed '{other.group(0).upper()}'.",
                          f"This task is '{MODULE_ARTICLE}'.",
                          f"{len(page['words'])} words and {len(page['points'])} "
                          f"points were written, all neatly done.",
                          "Nothing has been marked down."]}

    if not page["words"]:
        return {"case": "not homework",
                "facts": ["The page has no English-Hindi word list.",
                          "Nothing has been marked."]}

    # Mark the words. Unclear ones are left out, never marked wrong.
    right, wrong, unclear, unknown = [], [], [], []
    for w in page["words"]:
        expected = MODULE_WORDS.get(tidy(w["english"]))
        if expected is None:
            unknown.append(w["english"])
        elif not w.get("clear", True):
            unclear.append(w["english"])
        elif tidy(w["hindi"]) == tidy(expected):
            right.append(w["english"])
        else:
            wrong.append(w["english"])

    facts = [f"{len(page['words'])} words written."]
    if right or wrong:
        facts.append(f"{len(right)} of {len(right) + len(wrong)} checked answers correct.")
    if wrong:
        facts.append("To look at again: " + ", ".join(wrong) + ".")
    if unclear:
        facts.append("Too unclear to mark, so left out: " + ", ".join(unclear) + ".")
    if unknown:
        facts.append(f"{len(unknown)} words are not in the part of the key we hold.")
    facts.append(f"We only have {MODULE_WORDS_KNOWN} of {MODULE_WORDS_TOTAL} "
                 f"words, so no completeness percentage is given.")

    return {"case": "graded", "facts": facts}


result = check(page)

print(result["case"].upper())
for f in result["facts"]:
    print(" -", f)

## 5. Write the feedback  ·  **AI, needs the key**

The agent gets the findings from step 4 - not the photo, not the answer key. It
cannot change anything; it only turns findings into sentences.

In [ ]:
INSTRUCTIONS = """
You write homework feedback for children in India learning English.

You are given findings about one page. The checking is already done - you cannot
change it and must not mark anything yourself.

Write 2-3 short sentences in English, then the same message in Hindi.

- Say one specific thing they did well first, then one thing to do next.
- Warm, never harsh. Plain words.
- Talk about the work, never about the child. Say nothing about their effort,
  ability, attitude or home.
- If the page was a different module's homework: make clear they did nothing
  wrong, their work has not been marked down, and they should ask their teacher
  to move it to the right task.
- Do not invent any number that is not in the findings.
"""

writer = Agent(name="Feedback writer", instructions=INSTRUCTIONS)

reply = await Runner.run(writer, json.dumps(result, ensure_ascii=False))

print(reply.final_output)

In [1]:
import os, requests
from dotenv import load_dotenv
load_dotenv()

r = requests.get(os.environ["SUPABASE_URL"] + "/rest/v1/",
                 headers={"apikey": os.environ["SUPABASE_KEY"]})
spec = r.json()

tables = spec.get("definitions") or spec.get("components", {}).get("schemas", {})
for name, info in sorted(tables.items()):
    cols = list(info.get("properties", {}))
    print(f"{name}  ({len(cols)})")
    print("   ", ", ".join(cols))
    print()

In [2]:
print("status:", r.status_code)
print("top-level keys:", list(spec.keys()))
print()
print(str(spec)[:1500])

status: 401
top-level keys: ['message', 'hint']

{'message': 'Secret API key required', 'hint': 'Only secret API keys can be used for this endpoint.'}


In [3]:
select table_name, column_name, data_type
from   information_schema.columns
where  table_schema = 'public'
order  by table_name, ordinal_position;

SyntaxError: invalid syntax (2690334913.py, line 1)